# KING2 AI - Colab Delivery Notebook
---
**ضع ملفات التدريب في Google Drive تلقائياً**

شغّل الخلية أدناه مرتين:
1. المرة الأولى: يربط Drive ويتنزّل الملفات
2. المرة الثانية (اختياري): يشغّل `run_train.py`

In [ ]:
# @title 1. ربط Drive + تنزيل الملفات من GitHub
import os, subprocess, json
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create target folder
DRIVE_DIR = '/content/drive/MyDrive/Alking_AI_Train'
os.makedirs(DRIVE_DIR, exist_ok=True)

# GitHub raw URLs
BASE = 'https://raw.githubusercontent.com/MOT1209/mein-wep/main/models/king2%20ai-Model/Alking_AI_Train'
FILES = [
    'run_train.py',
    'king2_training.json',
    'dataset_info.json',
    'prepare_data.py',
]

print('جاري تنزيل الملفات من GitHub...')
for fname in FILES:
    url = f'{BASE}/{fname}'
    dst = os.path.join(DRIVE_DIR, fname)
    !wget -q "$url" -O "$dst"
    size = os.path.getsize(dst)
    print(f'  {fname} ({size:,} bytes) ✓')

print(f'\nجميع الملفات في {DRIVE_DIR}')
print('\nالآن شغّل الخلية الثانية لتشغيل التدريب')

---
## 2. تشغيل التدريب

شغّل الخلية أدناه لبدء التدريب مباشرة (QLoRA على Qwen2.5-3B-Instruct)

In [ ]:
# @title 2. تشغيل run_train.py
import os, subprocess
from google.colab import drive

DRIVE_DIR = '/content/drive/MyDrive/Alking_AI_Train'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Download run_train.py if not present
script = os.path.join(DRIVE_DIR, 'run_train.py')
if not os.path.exists(script):
    drive.mount('/content/drive')
    BASE = 'https://raw.githubusercontent.com/MOT1209/mein-wep/main/models/king2%20ai-Model/Alking_AI_Train'
    for fname in ['run_train.py', 'king2_training.json', 'dataset_info.json', 'prepare_data.py']:
        !wget -q "$BASE/$fname" -O "$DRIVE_DIR/$fname"
        print(f'{fname} downloaded ✓')
else:
    drive.mount('/content/drive')

os.chdir(DRIVE_DIR)
!python "{script}"

---
## 3. رفع الـ adapter إلى Hugging Face

بعد ما يخلص التدريب، شغّل الخلية أدناه لرفع `king2-qwen2.5-3b_adapter` على Hugging Face.

مطلوب: **Hugging Face token** من [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
# @title 3. رفع KING2 adapter إلى Hugging Face
import os
from getpass import getpass
from google.colab import drive

ADAPTER_DIR = '/content/drive/MyDrive/Alking_AI_Train/king2-qwen2.5-3b_adapter'
REPO_NAME = 'MOT1209/king2-qwen2.5-3b'  # غير لو بدك اسم مختلف

# Mount Drive
drive.mount('/content/drive')

# تأكد من وجود الـ adapter
if not os.path.exists(ADAPTER_DIR):
    raise FileNotFoundError(f'الـ adapter غير موجود: {ADAPTER_DIR}. شغّل التدريب أولاً.')

print('الملفات الموجودة:')
for f in os.listdir(ADAPTER_DIR):
    print(f'  {f}')

# طلب token
token = getpass('أدخل Hugging Face token: ')

# تسجيل الدخول
!huggingface-cli login --token "$token" --add-to-git-credential

# تثبيت huggingface_hub لو مش موجود
!pip install -q huggingface_hub

# رفع الـ adapter
from huggingface_hub import HfApi, create_repo

api = HfApi(token=token)

# إنشاء repo لو مش موجود
try:
    create_repo(REPO_NAME, private=False, exist_ok=True)
    print(f'Repo {REPO_NAME} جاهز ✓')
except Exception as e:
    print(f'ملاحظة: {e}')

# رفع الملفات
print('جاري رفع الملفات إلى Hugging Face...')
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO_NAME,
    repo_type='model',
    commit_message='رفع KING2 adapter - QLoRA on Qwen2.5-3B-Instruct',
)

print(f'\n✓ تم الرفع! https://huggingface.co/{REPO_NAME}')
usage = f'from peft import PeftModel\nmodel = PeftModel.from_pretrained(base_model, \"{REPO_NAME}\")'
print(f'\nاستخدم:\n{usage}')